In [ ]:
!pip install hdbscan

In [1]:
import os
# Работа с таблицами
import pandas as pd
import numpy as np

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt

# Предобработка данных
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.pipeline import Pipeline
from scipy import stats


# Уменьшение размерности
from sklearn.decomposition import PCA, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Алгоритмы кластеризации
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.ensemble import IsolationForest, \
                             RandomTreesEmbedding

from sklearn.model_selection import RandomizedSearchCV

from sklearn.mixture import GaussianMixture

# Метрики качества
from sklearn.metrics import accuracy_score, \
                            silhouette_score, \
                            davies_bouldin_score, \
                            silhouette_samples,\
                            calinski_harabasz_score

# Фиксация random_state
np.random.seed(42)

# Настройки графиков
sns.set(style='whitegrid')

plt.rcParams['figure.figsize'] = (10, 8)

# Загрузка и первичное исследование данных

In [ ]:
#df = pd.read_csv('/kaggle/working/clustering-physical-activity/Physical_Activity_Monitoring_unlabeled.csv')
df = pd.read_csv('/kaggle/input/competitions/clustering-physical-activity/Physical_Activity_Monitoring_unlabeled.csv')
print(df.shape)

df.head()

In [ ]:
def express_info(data, name='data', nans = True):
    '''Функция первичного исследования данных:
          - структура данных
          - пропуски
          - явные дубликаты
          - статистическое описание данных
       Аргументы:
          - data - датафрейм
          - name - название таблицы
          - nans флаг (True - результат пропущенных данных по столбцам)
    '''
    print(f'Размер данных ({name}):      {data.shape}')
    print(f'Количество явных дубликатов: {data.duplicated().sum()}')
    print(f'Наличие пропусков:           {data.isna().sum().sum()}')
    if nans:
        print(f'\nПропущенные данные:')
        #print(round(data.isna().mean()*100).sort_values(ascending=False).head(7), sep = '\n')
        print(data.isna().sum().sort_values(ascending=False).head(20), sep = '\n')        
    print(f'\nИнформация ({name}):')
    data.info()
    display(data.head(5))
    display(data.describe())
    print()

In [ ]:
express_info(df,  nans = True)

In [ ]:
# Статистика по subject_id
print("\nУчастники эксперимента:")
print(df['subject_id'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(10, 5))
df['subject_id'].value_counts().sort_index()\
                .plot(kind='bar', color='lightblue', edgecolor='black')
plt.xlabel('ID участника')
plt.ylabel('Количество записей')
plt.title('Распределение данных по участникам эксперимента')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Анализ временных меток
plt.figure(figsize=(12, 4))
plt.hist(df['timestamp'].diff().dropna(), bins=50, edgecolor='black')
plt.title('Распределение интервалов между измерениями')
plt.xlabel('Интервал (секунды)')
plt.show()

## Выводы:
- Размер данных (data):      (534601, 53)
- Количество явных дубликатов: 0
-  Пропуски в данных = 125732: много
    - Пропуски НЕ случайны: Все пропуски сконцентрированы в датчиках на руке (hand*)
    - У датчиков на груди и лодыжке пропусков значительно меньше или нет совсем, 
    скорее всего - человек мог снимать браслет с руки, но датчики на груди/лодыжке оставались.
    > Будем заполнять медианой по каждому `subject_id` отдельно (не по всем данным)

- Анализ временных меток `timestamp` показал:
    - Распределение нормальное.
    - Диапазон: от 37 до 4007 секунд (~1 час 7 минут)
    - Интервалы: есть отрицательные значения (ошибка синхронизации?)    
    - Разные субъекты имеют разные временные отрезки   
    - Это непрерывные временные ряды, а не независимые измерения!
- Дисбаланс участников (subject_id)
    - 1: 69,882 записей
    - 2: 68,740 записей
    - 3: 50,044 записей - меньше всех
    - 8: 73,047 записей - больше всех

## EDA

### Проверка на аномалии:

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Числовые признаки:", list(num_cols))

In [ ]:
# Выбросы для всех колонок
def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).sum()
    return outliers

# Получаем таблицу с количеством выбросов
outliers_count = df[num_cols].apply(detect_outliers_iqr).sort_values(ascending=False)
outliers_percent = (outliers_count / len(df)) * 100

outliers_summary = pd.DataFrame({
    'Колонка': outliers_count.index,
    'Кол-во выбросов': outliers_count.values,
    'Процент': outliers_percent.values
})

print("Топ-10 колонок с наибольшим количеством выбросов:")
print(outliers_summary.head(10))

### Обработка пропусков
#### по subject_id

In [ ]:
print(f"Пропусков до обработки: {df.isnull().sum().sum()}")

# Удаляем timestamp из обработки (не заполняем пропуски в нем)
timestamp_col = 'timestamp'
subject_col = 'subject_id'
feature_cols = [col for col in df.columns if col not in [timestamp_col, subject_col]]

# Заполняем пропуски медианой по каждому субъекту
df_filled = df.copy()
for subject in df[subject_col].unique():
    mask = df[subject_col] == subject
    for col in feature_cols:
        median_val = df.loc[mask, col].median()
        df_filled.loc[mask, col] = df_filled.loc[mask, col].fillna(median_val)

print(f"Пропусков после обработки: {df_filled.isnull().sum().sum()}")

Вывод:
- Видим массовые выбросы в гироскопах (30-37%). Скорее всего - это особенность физической активности(бег, прыжки, махи руками).
    > проблемные колонки: 'ankleGyro1', 'ankleGyro3', 'handGyro3', 'ankleGyro2', 'handGyro2', 'handGyro1', 'chestGyro3', 'chestGyro2', 'chestGyro1'
- это надо будет учесть при кластеризации
- строки и выбросы - не удаляем
- применяем RobustScaler в качестве стандартизации - сгладим выбросы
  > можно еще попробовать логарифмировть проблемные колонки (как 2 вариант набора данных).
- используем алгоритм, устойчивый к выбросам  - KMeans с нормализацией

## Формирование признаков

In [ ]:
# Делаем subject_id мультииндексом с timestamp
df_filled = df_filled.set_index(['subject_id', 'timestamp'])
print(f"Индекс: {df_filled.index.names}")
print(f"Форма: {df_filled.shape}")

### Набор логарифмировнных данных

In [ ]:
# Все колонки теперь признаки
X = df_filled

#problem_cols = ['ankleGyro1', 'ankleGyro3', 'handGyro3', 'ankleGyro2', 
#                'handGyro2', 'handGyro1', 'chestGyro3', 'chestGyro2', 'chestGyro1']
# Отделяем гироскопы (проблемные колонки)
gyro_cols = [col for col in X.columns if 'Gyro' in col]
other_cols = [col for col in X.columns if 'Gyro' not in col]
print(f"Гироскопов: {len(gyro_cols)}")
print(f"Остальных: {len(other_cols)}")
# Для гироскопов - логарифмическое преобразование для сжатия выбросов
#df_transformed = df.copy()
#for col in gyro_cols:
    # Добавляем константу, чтобы избежать log(0)
#    df_transformed[col] = np.log1p(np.abs(df[col])) * np.sign(df[col])
# 2 вариант логарифмическое преобразование
scaler_gyro = QuantileTransformer(output_distribution='normal')
X_gyro = scaler_gyro.fit_transform(X[gyro_cols])

# Для остальных - стандартное масштабирование
scaler_other = StandardScaler()
X_other = scaler_other.fit_transform(X[other_cols])

# Объединяем
X_scaled_log = np.hstack([X_gyro, X_other])

#### Набор данных стандартизированный RobustScaler()
RobustScaler - устойчив к выбросам

In [ ]:
# Набор данных стандартизированный RobustScaler()
#X = df_filled
#df_scaled = df.copy()
print(f"\nПризнаки: {list(X.columns)}")

scaler = RobustScaler()  # Один скейлер для всех признаков
X_scaled = scaler.fit_transform(X)

print(f"\nФорма после масштабирования: {X_scaled.shape}")

In [ ]:
# # Проверяем распределение
# plt.figure(figsize=(12, 4))
# plt.subplot(1, 2, 1)
# plt.hist(df['handGyro1'], bins=50, alpha=0.7)
# plt.title('До масштабирования')
# plt.subplot(1, 2, 2)
# plt.hist(df_scaled['handGyro1'], bins=50, alpha=0.7)
# plt.title('После RobustScaler')
# plt.show()

### Кластеризация
#### k-means

In [ ]:
def lokot_method(X, name = None):
    wcss = []
    K_values = range(1, 15)
    
    for k in K_values:
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(X)
        wcss.append(kmeans.inertia_)  # inertia_ -- сумма квадратов расстояний точек до центроидов
    
    # Строим график
    plt.plot(K_values, wcss, marker='o', linestyle='-')
    plt.xlabel("Число кластеров (K)")
    plt.ylabel("WCSS")
    plt.title(f"Метод локтя для выбора K ({name})")
    plt.show()

##### Проверяем разные наборы данных

In [ ]:
# Чистые данные
#lokot_method(X, 'X')
# Нормализованные данные RobustScaler
lokot_method(X_scaled, 'X_scaled')
# Нормализованные данные log+Standartscaler
#lokot_method(X_scaled_log, 'X_scaled_log')

In [ ]:
def siluet_method(X, sample_size= 30000, kvalues_list=[2, 3, 4, 5, 6], name="X"):
    sample_size = sample_size
    np.random.seed(42)
    if hasattr(X, 'iloc'):  # Если DataFrame
        sample_indices = np.random.choice(len(X), sample_size, replace=False)
        X_sample = X.iloc[sample_indices].values
    else:  # Если numpy array
        sample_indices = np.random.choice(len(X), sample_size, replace=False)
        X_sample = X[sample_indices] # используем .iloc для DataFrame

    print(f'\nТестируем набор данных: {name}')
    k_values = kvalues_list
    for k in k_values:
        print(f"Тестируем k={k}")
        kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=5000, n_init=10)
        labels = kmeans.fit_predict(X_sample)
        score = silhouette_score(X_sample, labels)
        print(f"K={k}: Silhouette={score:.4f}")


In [ ]:
%%time
siluet_method(X_scaled, 10000, [2, 3, 5, 8, 9, 10, 11, 12, 13, 14], 'X_scaled')
#siluet_method(X, 30000, [2, 3, 4, 5, 6], 'X')
#siluet_method(X_scaled_log, 30000, [2, 3, 4, 5, 6], 'X_scaled_log')

In [ ]:
# берем выборку
sample_size = 30000  # можно уменьшить до 15000 для скорости
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

In [ ]:
# Функция для построения силуэтных графиков (без изменений)
def plot_silhouette(X, cluster_counts):
    fig, axes = plt.subplots(1, len(cluster_counts), figsize=(18, 6))
    for idx, K in enumerate(cluster_counts):
        kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X)
        # Вычисляем силуэтные коэффициенты
        silhouette_vals = silhouette_samples(X, cluster_labels)
        # Средний силуэтный коэффициент
        avg_score = silhouette_score(X, cluster_labels)

        y_lower, y_upper = 0, 0
        axes[idx].set_title(f"K = {K}")
        for i in range(K):
            cluster_silhouette_vals = silhouette_vals[cluster_labels == i]
            cluster_silhouette_vals.sort()

            y_upper += len(cluster_silhouette_vals)
            axes[idx].fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                cluster_silhouette_vals,
                alpha=0.7
            )
            y_lower = y_upper

        # Статистика
        print(f"\nСтатистика для K={K}:")
        print(f"  Общий силуэт: {avg_score:.3f}")
        for i in range(K):
            print(f"  Кластер {i}: mean={silhouette_vals[cluster_labels==i].mean():.3f}, "
                      f"std={silhouette_vals[cluster_labels==i].std():.3f}, "
                      f"size={sum(cluster_labels==i):,}")

        # Средний силуэтный коэффициент
        #avg_score = silhouette_score(X, cluster_labels)
        axes[idx].axvline(avg_score, linestyle="--", color="red", label=f"Mean: {avg_score:.3f}")
        axes[idx].legend()
        axes[idx].set_xlabel("Silhouette Coefficient")
        axes[idx].set_ylabel("Cluster Size")

    plt.suptitle("Silhouette Plots for Different K", fontsize=14)
    plt.show()


# берем выборку
sample_size = 30000  # можно уменьшить до 15000 для скорости
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

# Вызываем функцию для K= 2, 3 на выборке
plot_silhouette(X_sample, cluster_counts=[2, 3, 5, 8, 9, 10, 11, 12, 13, 14])

Значения нестабильны - скачут, лучшее значение у 2 кластера, но тут может быть  более сложная структура. Проверяем значения с помощью иерархической кластеризации.

#### Иерархическая кластеризация
- AgglomerativeClustering

In [ ]:
# Для визуализации
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print('Размер после PCA:')
print(X_pca.shape)


#### Построим Дендограмму

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

sample_size_d = 5000 
X_sample_d = X[:sample_size_d]
# Строим матрицу расстояний (linkage matrix)
Z = linkage(X_sample_d, method="average")  # Метод average для минимизации дисперсии

# Строим дендрограмму
plt.figure(figsize=(20, 15))
dendrogram(
    Z, 
    p=150,                  # Показываем 150 листьев
    truncate_mode='lastp',  # Обрезаем до p листьев
    leaf_rotation=90.,
    leaf_font_size=8.,
    show_leaf_counts=True
)
plt.title("Дендрограмма для агломеративной кластеризации (первые 150 объектов)")
plt.xlabel("Индекс объекта")
plt.ylabel("Расстояние между кластерами")
plt.show()

Вывод:
 - По структуре ключевое количество кластеров может быть: 2, 5, 6, 8, 9, 10, 11, 12, 13 - считаю это оптимальным разделением, иначе это будет уже излишне большое разделение.
 - 2 кластера, как ранее предполагалось слишком грубое разделение

In [ ]:
# Пробуем разные методы связи
linkage_methods = ['ward', 'complete', 'average', 'single']
sample_size = 10000
X_sample = X_scaled[:sample_size]

print(f"Сравнение методов иерархической кластеризации sample_size = {sample_size}:")
print("=" * 50)

for linkage in linkage_methods:
    for k in [6, 8, 9, 10, 11, 12]:
        hier = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels = hier.fit_predict(X_sample)
        score = silhouette_score(X_sample, labels)
        print(f"linkage={linkage}, k={k}: silhouette={score:.4f}")
    print("-" * 50)

#### Вывод:
- лучше всего показал себя метод связи linkage=average, совсем немного хуже linkage=single,
- лучшие данные по всем связям при k = 6, 8, 9, 10:
    -  linkage=average, k=6: silhouette=0.6193
    -  linkage=average, k=8: silhouette=0.6047
    -  linkage=average, k=9: silhouette=0.5944
    -  linkage=average, k=10: silhouette=0.5850
    > Начиная с 11 - идет снижение качества
- будем обучать модель на linkage=average, k = 6, 8, 9, 10, проверим больший размер данных sample_size = 30000
- возможно 6 - слишком малое разделение.

In [ ]:
# Иерархическая кластеризация (может дать лучшую структуру)
def aggl_cluster(X, name = 'X', n_clusters=2, name_method='Agglomerative_Clustering',\
                 sample_size = 30000, grafh=False):
    #sample_size = sample_size
    X_sample = X[:sample_size]

    print(f'Кластеризаци на наборе данных "{name}":')
    hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage="average")
    labels_hier = hierarchical.fit_predict(X_sample)
    score_hier = silhouette_score(X_sample, labels_hier)
    print(f"\n{name_method}: silhouette_score = {score_hier:.4f}")
    if grafh:
        # PCA для визуализации
        pca = PCA(n_components=n_clusters)
        X_pca = pca.fit_transform(X)
        plt.scatter(
            X_pca[:sample_size, 0],
            X_pca[:sample_size, 1],
            c=labels_hier, cmap='viridis',
            alpha=0.6, s=5, edgecolors='none'
            )
        plt.title(f'{name_method}: n_clusters={n_clusters}, score={score_hier:.4f}')    
        plt.xlabel('PCA 1')
        plt.ylabel('PCA 2')      
        plt.show()
    return labels_hier, score_hier
#labels_hier_sc, score_hier_sc = aggl_cluster(X_scaled, 'X_scaled', 2, grafh=True)

In [ ]:
labels_hier_sc, score_hier_sc = aggl_cluster(X_scaled, 'X_scaled', 8, grafh=True)

In [ ]:
labels_hier_sc, score_hier_sc = aggl_cluster(X_scaled, 'X_scaled', 9, grafh=True)

In [ ]:
labels_hier_sc, score_hier_sc = aggl_cluster(X_scaled, 'X_scaled', 10, grafh=True)

Вывод:
- На большей выборке score_hier улучшается, возможно наше предположение верно:
- лучший результат k=8, silhouette_score = 0.6459

In [ ]:
def grafh_pca(X, name, labels,  n_clusters, sample_size,  name_method):
    # PCA для визуализации
    pca = PCA(n_components=n_clusters)
    X_pca = pca.fit_transform(X)
    plt.scatter(X_pca[:sample_size, 0], X_pca[:sample_size, 1],
                c=labels, cmap='viridis', alpha=0.6, s=5, edgecolors='none'
            )
    plt.title(f'{name_method}: n_clusters = {n_clusters}')
    
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')
    
    plt.show()
#grafh_pca(X_scaled, 'X_scaled', labels_hier_sc, 2, sample_size,'Agglomerative Clustering')

#### Вывод:
- лучший результат дал набор данных X_scaled - будем работать с этим набором данных;
- хуже результат показали чистые данные, плохие результаты дал набор данных X_scaled_log
- метод локтя указывает на 2-3 кластера;
- Метод силуэта на части данных X_scaled показал K=2:
    - Silhouette=0.53(MiniBatchKMeans)
    - Иерархическая кластеризация: Silhouette=0.54
- Но дендраграмма показала более сложную структуру данных, поэтому деление на 2 кластера может оказаться грубым, что подтверждают исследования:
  - если смотрим 2 кластера: по структуре - 1 кластер большой и хороший, 0 - кластер очень маленький и плохой.
- посмотрела исходник (в датасете PAMAP2 12 различных активностей - потому, даже если хороший результат на 2 кластеров, будем смотреть и на большее количество), этот момент тоже учтем.
- Проверим ключевые кластеры k=2,5,8,9,10,11,12;
- Проверим разные алгоритмы: KMeans и AgglomerativeClustering, в первую очередь, проверим DBSCAN - посмотрим какое количество кластеров он подберет, HDBSCAN (если необходимо)

## Обучение
- берем для обучения X_scaled
- k=2,5,8,9,10,11,12;

### KMeans и Agglomerative

In [ ]:
sample_size = 20000
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

In [ ]:
%%time
# Упрощенная версия для быстрого сравнения
sample_size = 15000
X_sample = X_scaled[np.random.choice(len(X_scaled), sample_size, replace=False)]

models = {
    'KMeans (k=8)': KMeans(n_clusters=8, random_state=42, n_init=10),
    'KMeans (k=10)': KMeans(n_clusters=10, random_state=42, n_init=10),
    'KMeans (k=12)': KMeans(n_clusters=12, random_state=42, n_init=10),
    #'MiniBatchKMeans (k=2)': MiniBatchKMeans(n_clusters=2, random_state=42,\
     #                                        batch_size=5000, n_init=10),
    'MiniBatchKMeans (k=8)': MiniBatchKMeans(n_clusters=8, random_state=142,\
                                             batch_size=5000, n_init=10),
    'MiniBatchKMeans (k=10)': MiniBatchKMeans(n_clusters=10, random_state=142,\
                                             batch_size=5000, n_init=10), #, max_iter=300
    'MiniBatchKMeans (k=11)': MiniBatchKMeans(n_clusters=11, random_state=42,\
                                             batch_size=5000, n_init=10),
    'Agglomerative (k=8)': AgglomerativeClustering(n_clusters=8, linkage='average'),
    'Agglomerative (k=10)': AgglomerativeClustering(n_clusters=10, linkage='average'),
}

print("Сравнение моделей (Silhouette Score):")
print("-" * 40)

for name, model in models.items():
    labels = model.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    n_clusters = len(set(labels))
    print(f"{name:25}:  Score = {score:.4f}, Кластеров: {n_clusters}")

##### Вывод:
- K = 5 - лучшее значение, но мало кластеров
- K=10 и K=8 - хороший  результат на KMeans + оптимальное число кластеров:
  - k=10:  Score = 3901
  - k=8: Score = 0.4883 - более стабильный хороший результат
 > В датасете PAMAP2 (откуда взяты данные) реально 12 различных активностей:
   - K=10 приближается к реальному числу активностей!
- Лучшие результаты показывает себя Agglomerative: Score = 0.6444 и Score = 0.6330 - это уже иерархическая модель, посмотрим что подберет DBSCAN.

### DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Ищем расстояния до 20 ближайших соседей
n_neighb=20
nbrs = NearestNeighbors(n_neighbors=n_neighb).fit(X_sample)
distances, _ = nbrs.kneighbors(X_sample)

# Берем 5-е расстояние, сортируем и строим график
distances = np.sort(distances[:, 4])
plt.plot(distances)
plt.xlabel("Точки данных")
plt.ylabel(f"Расстояние до {n_neighb} ближайшего соседа")
plt.title("Выбор eps для DBSCAN")
plt.show()

# На графике ищем "излом" - это и есть оптимальное значение eps.

 Вывод:
 - Метод ближайших соседей показывает eps = (3.0 - 4.5) в области перегиба 

In [ ]:
print("Подбор лучших параметров DBSCAN:")
# Тестируем разные комбинации параметров eps в области перегиба (3.0 - 4.5)
param_grid = [
    # eps в области перегиба (3.0 - 4.5)
    (3.0, 50),(3.0, 75),(3.0, 100),(3.0, 150),    
    (3.2, 50),(3.2, 75),(3.2, 100),    
    (3.5, 50),(3.5, 75), (3.5, 100),(3.5, 150),    
    (3.8, 50),(3.8, 75), (3.8, 100),    
    (4.0, 50), (4.0, 75), (4.0, 100),    
    (4.2, 50), (4.2, 75), (4.2, 100),    
    (4.5, 50), (4.5, 75), (4.5, 150),    
    # Контрольные точки (границы)
    (2.5, 100), (5.0, 100),
]
for eps, min_samples in param_grid:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
    labels = dbscan.fit_predict(X_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = sum(labels == -1) / len(labels) * 100
    
    if n_clusters > 0:
        print(f"eps={eps}, min_samples={min_samples}: "
              f"кластеров={n_clusters}, шум={noise_ratio:.1f}%")

In [ ]:
# Выбираем оптимальные параметры (например, eps=3.5, min_samples=50)
#k=10 # шум=54.2%
#optimal_eps = 3.2, optimal_min_samples = 50
#k=8 # шум=53.2%
optimal_eps=3.5 
optimal_min_samples=50

dbscan_opt = DBSCAN(eps=optimal_eps, min_samples=optimal_min_samples, n_jobs=-1)
labels_opt = dbscan_opt.fit_predict(X_sample)

n_clusters = len(set(labels_opt)) - (1 if -1 in labels_opt else 0)
noise_count = sum(labels_opt == -1)
noise_ratio = noise_count / len(labels_opt) * 100

print(f"\nОптимальные параметры: eps={optimal_eps}, min_samples={optimal_min_samples}")
print(f"Найдено кластеров: {n_clusters}")
print(f"Шум: {noise_count} ({noise_ratio:.1f}%)")

# Статистика по кластерам
print("\nСтатистика кластеров:")
for label in sorted(set(labels_opt)):
    if label == -1:
        print(f"  Шум: {sum(labels_opt == label)} точек")
    else:
        print(f"  Кластер {label+1}: {sum(labels_opt == label)} точек")

In [ ]:
# PCA для визуализации
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sample)

#plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                      cmap='viridis', 
                      c=labels_opt, alpha=0.6, s=5, edgecolors='none')
plt.title(f'DBSCAN Кластеризация (eps={optimal_eps}, min_samples={optimal_min_samples})\n'
          f'Кластеров: {n_clusters}, Шум: {noise_ratio:.1f}%', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### HDBSCAN 

In [ ]:
# print("\nАльтернатива: HDBSCAN (может дать лучшие результаты)")

# import hdbscan
# #X_sample = X_scaled[sample_indices]
# # HDBSCAN автоматически определяет количество кластеров
# hdb = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50, prediction_data=True)
# labels_hdb = hdb.fit_predict(X_sample)

# n_clusters_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
# noise_ratio_hdb = sum(labels_hdb == -1) / len(labels_hdb) * 100

# print(f"HDBSCAN результаты:")
# print(f"  Кластеров: {n_clusters_hdb}")
# print(f"  Шум: {sum(labels_hdb == -1):,} ({noise_ratio_hdb:.1f}%)")

#### Вывод:
Для DBSCAN на данных 20000:
- Лучшие параметры: eps = 3.5, min_samples = 50
   - Дает ровно 8 кластеров
   - Шум 53.2%
- Проблема DBSCAN, HDBSCAN  - не подходят для данных:
   - Шум > 50% - неприемлемо (более 50% данных остаются неклассифицированными)
   - Нет четкой кластерной структуры (точки распределены непрерывно)
   - Разная плотность активностей
- возвращаемся к KMeans - быстрый, легко интерпретируемый метод

In [ ]:
comparison = {
    'Метод': [
                'Agglomerative (average, k=8)', 
                'Agglomerative (average, k=10)',
                'MiniBatchKMeans (k=8)',
                'KMeans (k=10)',
                'DBSCAN (eps=3.2, min_samples=50)'
            ],
    'Кластеров': [8, 10, 8, 10, 8],
    'Шум %': [0, 0, 0, 0, 54.2],
    'Силуэт': [0.6444, 0.6330, 0.4883, 0.3820, 'N/A'],
    'Пригодность': ['Отлично', 'Отлично', 'Очень хорошо', 'Хорошо', '50% потерь']
}

df_compare = pd.DataFrame(comparison)
print(df_compare.to_string(index=False))

### Вывод:

- K=10 - лучший результат на KMeans:
  - На финалке проверим:
    > MiniBatchKMeans (k=8)': MiniBatchKMeans(n_clusters=10, random_state=142,batch_size=5000, n_init=10)
- также смотрим Agglomerative (k=10), но тут вопрос в ресурсах - проверим.
  - Для финального решения:
   >  берем AgglomerativeClustering (linkage='average', n_clusters=10)\
      Это даст 100% точек в кластерах и силуэт 0.5850
- DBSCAN при eps = 3.5, min_samples = 50 стабильно определяет k=8 и показала низкий уровень шума:
- но DBSCAN  - не подходит для наших данных.
   - Шум > 50% - неприемлемо (более 50% данных остаются неклассифицированными)
   - Нет четкой кластерной структуры
- все 3 метода хорошо повели себя при k=8 и k=10, потому будем тестировать на 10 кластерах.

## Проверка лучшей модели, подготовка к сабмиту

### kmeans

In [ ]:
%%time
print(" Кластеризация:")

kmeans_final = MiniBatchKMeans(n_clusters=8, random_state=142, batch_size=5000, n_init=20, max_iter=300)
clusters = kmeans_final.fit_predict(X_scaled)
print(f"   Кластеризация завершена. Кластеров: {len(set(clusters))}")


In [ ]:
# %%time
# #не подходит - на батчах лучше обучать
# kmeans_final = KMeans( n_clusters=8, random_state=142, 
#                         n_init=20,          # критический параметр
#                         max_iter=200,        # Уменьшаю (100-200 оптимально)
#                         algorithm='lloyd'    # Классический алгоритм (более стабильный)
#                     )
# clusters = kmeans_final.fit_predict(X_scaled)

In [ ]:
#clusters = clusters_kmeans.copy()
# Добавляем кластеры в DataFrame
df_filled['cluster'] = clusters
df_filled.sample(5)

In [ ]:
print("Визуализация:")
# Берем выборку для визуализации
sample_size = min(100000, len(X_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled[sample_idx])

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                       c=clusters[sample_idx], cmap='coolwarm', alpha=0.5, s=1)
plt.colorbar(scatter, label='Кластер')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'Визуализация кластеров (PCA): K={len(set(clusters))}')
plt.show()

# print("Анализ результатов:")
# cluster_sizes = df_filled['cluster'].value_counts().sort_index()
# print(f"   Распределение по кластерам:")
# for i in range(9):
#     print(f"     Кластер {i}: {cluster_sizes[i]:,} ({cluster_sizes[i]/len(df_filled)*100:.1f}%)")

# # Проверка независимости от субъектов
# print(f"\n   Распределение по субъектам (%):")
# cross_tab = pd.crosstab(df_filled.index.get_level_values('subject_id'), 
#                          df_filled['cluster'], normalize='index')
# print(cross_tab.round(3))

### Подготовка файла для сабмита

In [ ]:
print(f"\nФинальная кластеризация с K={len(set(clusters))}")
submission = pd.DataFrame({
    'Index': np.arange(len(df_filled)),
    'activityID': clusters + 1
})

submission.to_csv('submission_kb8.csv', index=False)
print("\nСабмит создан: submission_kb8.csv")
print(submission['activityID'].value_counts().sort_index())

In [ ]:
print('таблица загрузок сабмитов с k=2,8,10,11')
data = {
    'K': [2, 8, 10, 11],
    'Silhouette': [0.4322, 0.4883, 0.1000, 0.2647],
    'Kaggle_Score': [0.18076, 0.37901, 0.35849, 0.12310],
    'Оценка': ['Слишком грубое разделение', \
               'Более стабильный результат', \
               'Соответствует реальности, но менее стабильный результат',\
               'Слишком много кластеров (переизмельчение)']
}
table = pd.DataFrame(data)
display(table)

# Сохраняем
#df.to_csv('my_results.csv', index=False)
#print("\nТаблица сохранена в my_results.csv")

#### Сохраняем

In [ ]:
submission.to_csv('submission.csv', index=False)
print(f"   Файл submission.csv создан!")
print(f"   Формат: {submission.shape[0]} строк, колонки: {list(submission.columns)}")
print(f"   Кластеры в сабмите: {sorted(submission['activityID'].unique())}")
print("Файл для отправки: submission.csv")

### AgglomerativeClustering

In [ ]:
%%time
print("Кластеризация AgglomerativeClustering (k=10):")
print("Это может занять ВРЕМЯ - ждем)")

# Используем выборку 100k для ускорения (если памяти мало)
sample_size = 50000
linkage1='average'
if len(X_scaled) > sample_size:
    print(f"   Используем выборку sample_size = {sample_size}, linkage = {linkage1}:")
    np.random.seed(42)
    sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)
    X_sample = X_scaled[sample_idx]
    
    # Обучаем на выборке
    hier = AgglomerativeClustering(n_clusters=8, linkage=linkage1)
    labels_sample = hier.fit_predict(X_sample)
    print(f"   Кластеров получено: {len(np.unique(labels_sample))}")
    
    # Обучаем классификатор для предсказания на всех данных
    from sklearn.ensemble import RandomForestClassifier
    print("   Обучаем классификатор RandomForestClassifier для всех данных:")
    #clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf = RandomForestClassifier(n_estimators=100,
                                random_state=142,
                                n_jobs=-1,
                                max_depth=20,
                                verbose=0
                                )
    clf.fit(X_sample, labels_sample)
    print(" Предсказание для всех данных")
    clusters = clf.predict(X_scaled)
else:
    # Если данных немного - кластеризуем все
    hier = AgglomerativeClustering(n_clusters=8, linkage=linkage1)
    clusters = hier.fit_predict(X_scaled)

print(f"   Готово! Кластеров: {len(np.unique(clusters))}")

# Проверка распределения
print("\n6. Распределение по кластерам:")
unique, counts = np.unique(clusters, return_counts=True)
for i, (c, cnt) in enumerate(zip(unique, counts)):
    print(f"   Кластер {c}: {cnt:,} ({cnt/len(clusters)*100:.1f}%)")


# Финальная кластеризация
print(f"\nФинальная кластеризация с K={len(set(clusters))}")
submission = pd.DataFrame({
    'Index': np.arange(len(df_filled)),
    'activityID': clusters + 1
})
submission.to_csv('submission_aa10.csv', index=False)
print("\nСабмит создан: submission_k5.csv")
print(submission['activityID'].value_counts().sort_index())

#Сохраняем
submission.to_csv('submissiona_aa8.csv', index=False)
print(f"   Файл submission.csv создан!")
print(f"   Формат: {submission.shape[0]} строк, колонки: {list(submission.columns)}")
print(f"   Кластеры в сабмите: {sorted(submission['activityID'].unique())}")
print("Файл для отправки: submission_aa8.csv")

### Вывод:
- Очень низкое значение на тесте AgglomerativeClustering: 0.18063
    - получили один гигантский кластер (99.8% данных) и 9 крошечных.
      > очевидно, что на такой выборке алгоритм AgglomerativeClustering не смог обобщить структуру, больше выборку ресурс не позволяет сделать
- Хорошо в итоге показала себя модель Kmeans с k = 8 и k =10, особенно выигрывает по времени 1 мин против 10 мин.
- потому приходим к итоговому **Kmeans: 0.37901, k = 8**

In [ ]:
data = {'Модели': ['MiniBatchKMeans',
                    'MiniBatchKMeans',
                    'MiniBatchKMeans',
                    'MiniBatchKMeans',
                    'KMeans',
                    'Agglomerative (average)',
                    'Agglomerative (ward)',
                    'Agglomerative (average)'],
    'K': [2, 8, 10, 11, 8, 10, 10, 8],
    'Silhouette': [0.4322, 0.4883, 0.1000, 0.2647, 0.4144, 0.6330, 'N/A', 0.6444],
    'Kaggle_Score': [0.18076, 0.37901, 0.35849, 0.12310, 0.27357, 0.18063, 0.17833, 0.15358 ],
    'Примечание': ['Слишком грубое разделение',
                    'ЛУЧШИЙ РЕЗУЛЬТАТ - стабильный показатель',
                    'Хороший результат - правдоподобное количество кластеров',
                    'Слишком много кластеров (переизмельчение)',
                    'Неплохой результат - но хуже чем на батчах',
                    '1 кластер = 99.8% данных (дисбаланс)',
                    '1 кластер = 99.8% данных (дисбаланс)',
                    '99.9% в одном кластере-отсутствие кластеризации']}
table = pd.DataFrame(data)
print("Таблица результатов кластеризации:")
display(table)